# NCD Risk Factor Analysis — NHANES 2017-2018
**Author:** Afriyie Karikari Bempah, PharmD  
**Data:** CDC NHANES 2017-2018 (Demographics, Blood Pressure, BMI, Diabetes)  
**Tools:** Python, pandas, scikit-learn, scipy, matplotlib

---

### Research Questions
1. What are the prevalence rates of diabetes, hypertension, and obesity in the US population?
2. Which factors most strongly predict diabetes risk — and what are the odds ratios?
3. Where do the biggest racial/ethnic health disparities lie?

### Clinical Relevance
Non-communicable diseases (NCDs) account for 74% of global deaths. Understanding their risk factors and health equity dimensions is foundational to population health strategy — including for emerging NCD burdens in African markets.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

## 1. Load & Merge NHANES Data

In [ ]:
# load locally downloaded NHANES 2017-2018 files
demo = pd.read_sas("DEMO_J.XPT", format='xport', encoding='utf-8')
bpx  = pd.read_sas("BPX_J.XPT",  format='xport', encoding='utf-8')
bmx  = pd.read_sas("BMX_J.XPT",  format='xport', encoding='utf-8')
diq  = pd.read_sas("DIQ_J.XPT",  format='xport', encoding='utf-8')

print(f"Demographics: {demo.shape}")
print(f"Blood pressure: {bpx.shape}")
print(f"Body measurements: {bmx.shape}")
print(f"Diabetes: {diq.shape}")

# merge on SEQN (participant ID)
df = demo.merge(bpx, on='SEQN', how='inner')         .merge(bmx, on='SEQN', how='inner')         .merge(diq, on='SEQN', how='inner')

# select and rename variables
cols = {
    'SEQN': 'ID', 'RIDAGEYR': 'Age', 'RIAGENDR': 'Gender',
    'RIDRETH3': 'Race_Ethnicity', 'INDFMPIR': 'Poverty_Income_Ratio',
    'BPXSY1': 'Systolic_BP', 'BPXDI1': 'Diastolic_BP',
    'BMXBMI': 'BMI', 'BMXWAIST': 'Waist_Circumference',
    'DIQ010': 'Diabetes_Diagnosis'
}
df = df[list(cols.keys())].rename(columns=cols)

# create outcome variables
df['Diabetes']     = (df['Diabetes_Diagnosis'] == 1).astype(int)
df['Hypertension'] = ((df['Systolic_BP'] >= 130) | (df['Diastolic_BP'] >= 80)).astype(int)
df['Obese']        = (df['BMI'] >= 30).astype(int)

df_clean = df.dropna(subset=['Age','BMI','Systolic_BP','Poverty_Income_Ratio','Diabetes'])

print(f"\nClean dataset: {df_clean.shape}")
print(f"Diabetes prevalence:    {df_clean['Diabetes'].mean()*100:.1f}%")
print(f"Hypertension prevalence:{df_clean['Hypertension'].mean()*100:.1f}%")
print(f"Obesity prevalence:     {df_clean['Obese'].mean()*100:.1f}%")

## 2. Decode Demographics & Summary Statistics

In [ ]:
race_map = {1:'Mexican American', 2:'Other Hispanic', 3:'Non-Hispanic White',
            4:'Non-Hispanic Black', 6:'Non-Hispanic Asian', 7:'Other/Multiracial'}
gender_map = {1:'Male', 2:'Female'}

df_clean = df_clean.copy()
df_clean['Race_Label']   = df_clean['Race_Ethnicity'].map(race_map)
df_clean['Gender_Label'] = df_clean['Gender'].map(gender_map)
df_clean['Age_Group']    = pd.cut(df_clean['Age'], bins=[0,18,35,50,65,100],
                                   labels=['<18','18-35','35-50','50-65','65+'])

print("Summary statistics:")
print(df_clean[['Age','BMI','Systolic_BP','Poverty_Income_Ratio']].describe().round(2))

print("\nRace/ethnicity distribution:")
print(df_clean['Race_Label'].value_counts())

## 3. EDA — NCD Prevalence by Demographics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax_idx, (ncd, race_data) in enumerate([
    ('Diabetes', df_clean.groupby('Race_Label')['Diabetes'].mean()*100),
    ('Hypertension', df_clean.groupby('Race_Label')['Hypertension'].mean()*100),
    ('Obese', df_clean.groupby('Race_Label')['Obese'].mean()*100)
]):
    data = race_data.sort_values(ascending=False)
    axes[0,ax_idx].bar(range(len(data)), data.values, color='steelblue')
    axes[0,ax_idx].set_xticks(range(len(data)))
    axes[0,ax_idx].set_xticklabels(data.index, rotation=30, ha='right', fontsize=8)
    axes[0,ax_idx].set_title(f'{ncd} Prevalence by Race/Ethnicity (%)')
    axes[0,ax_idx].set_ylabel('Prevalence (%)')
    for i, v in enumerate(data.values):
        axes[0,ax_idx].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=8)

age_diab = df_clean.groupby('Age_Group', observed=True)['Diabetes'].mean()*100
axes[1,0].bar(age_diab.index, age_diab.values, color='steelblue')
axes[1,0].set_title('Diabetes Prevalence by Age Group (%)')
axes[1,0].set_ylabel('Prevalence (%)')

axes[1,1].hist(df_clean[df_clean['Diabetes']==0]['BMI'],
               bins=40, alpha=0.6, color='steelblue', label='No Diabetes')
axes[1,1].hist(df_clean[df_clean['Diabetes']==1]['BMI'],
               bins=40, alpha=0.6, color='orange', label='Diabetes')
axes[1,1].set_title('BMI Distribution by Diabetes Status')
axes[1,1].set_xlabel('BMI')
axes[1,1].legend()

axes[1,2].boxplot([
    df_clean[df_clean['Diabetes']==0]['Poverty_Income_Ratio'].dropna(),
    df_clean[df_clean['Diabetes']==1]['Poverty_Income_Ratio'].dropna()
], labels=['No Diabetes','Diabetes'])
axes[1,2].set_title('Poverty Income Ratio by Diabetes Status')
axes[1,2].set_ylabel('Poverty Income Ratio')
axes[1,2].grid(alpha=0.3)

plt.suptitle('NCD Risk Factor Analysis — NHANES 2017-2018', fontsize=14)
plt.tight_layout()
plt.savefig('chart1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight: The most striking disparity is hypertension in Non-Hispanic Black
# patients (41%) — 12 points above the population average.
# Age remains the dominant diabetes risk factor — 65+ prevalence is 29%.

## 4. Logistic Regression — Diabetes Prediction

In [ ]:
features = ['Age','BMI','Systolic_BP','Poverty_Income_Ratio','Gender','Race_Ethnicity']

X = df_clean[features].copy()
y = df_clean['Diabetes']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['Age','BMI','Systolic_BP','Poverty_Income_Ratio']] =     scaler.fit_transform(X[['Age','BMI','Systolic_BP','Poverty_Income_Ratio']])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# standard model
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_proba = lr.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)

# balanced model
lr_bal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_bal.fit(X_train, y_train)
y_pred_bal = lr_bal.predict(X_test)
y_pred_proba_bal = lr_bal.predict_proba(X_test)[:,1]
auc_bal = roc_auc_score(y_test, y_pred_proba_bal)

print(f"Standard Model AUC: {auc:.3f}")
print(f"Balanced Model AUC: {auc_bal:.3f}")
print("\nBalanced Model Classification Report:")
print(classification_report(y_test, y_pred_bal,
      target_names=['No Diabetes','Diabetes']))

## 5. Odds Ratios & Forest Plot

In [ ]:
coefficients = lr_bal.coef_[0]
odds_ratios  = np.exp(coefficients)
ci_lower = np.exp(coefficients - 1.96 * np.sqrt(np.diag(np.linalg.inv(X_train.T @ X_train))))
ci_upper = np.exp(coefficients + 1.96 * np.sqrt(np.diag(np.linalg.inv(X_train.T @ X_train))))

or_df = pd.DataFrame({
    'Feature': ['Age','BMI','Systolic BP','Poverty Ratio','Gender (Male)','Race/Ethnicity'],
    'Odds_Ratio': odds_ratios,
    'CI_Lower': ci_lower,
    'CI_Upper': ci_upper
}).sort_values('Odds_Ratio', ascending=False)

print("Odds Ratios for Diabetes Risk Factors:")
print(or_df.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_or = ['#F44336' if v > 1 else '#4CAF50' for v in or_df['Odds_Ratio']]
axes[0].barh(or_df['Feature'], or_df['Odds_Ratio'], color=colors_or, alpha=0.8)
axes[0].errorbar(
    or_df['Odds_Ratio'], or_df['Feature'],
    xerr=[or_df['Odds_Ratio']-or_df['CI_Lower'],
          or_df['CI_Upper']-or_df['Odds_Ratio']],
    fmt='none', color='black', capsize=5, linewidth=2
)
axes[0].axvline(x=1, color='black', linestyle='--', linewidth=1.5, label='OR=1 (No Effect)')
axes[0].set_xlabel('Odds Ratio (95% CI)')
axes[0].set_title('Odds Ratios for Diabetes Risk Factors\n(Red=Risk | Green=Protective)')
axes[0].legend()
axes[0].grid(alpha=0.3)

fpr1, tpr1, _ = roc_curve(y_test, y_pred_proba)
fpr2, tpr2, _ = roc_curve(y_test, y_pred_proba_bal)
axes[1].plot(fpr1, tpr1, color='steelblue', label=f'Standard LR (AUC={auc:.3f})')
axes[1].plot(fpr2, tpr2, color='orange', label=f'Balanced LR (AUC={auc_bal:.3f})')
axes[1].plot([0,1],[0,1],'k--',alpha=0.3,label='Random Baseline')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Diabetes Risk Factor Analysis — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.savefig('chart2_odds_ratios.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight: Age (OR=4.67) and BMI (OR=1.79) are dominant risk factors.
# The forest plot format mirrors published NHANES analyses in
# journals like Diabetes Care and JAMA.

## 6. Health Equity Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
race_order = ['Non-Hispanic Black','Mexican American','Other Hispanic',
              'Non-Hispanic White','Other/Multiracial','Non-Hispanic Asian']

for i, (ncd, title) in enumerate([
    ('Diabetes','Diabetes Prevalence (%)'),
    ('Hypertension','Hypertension Prevalence (%)'),
    ('Obese','Obesity Prevalence (%)')
]):
    vals = df_clean.groupby('Race_Label')[ncd].mean()*100
    vals = vals.reindex(race_order)
    avg = vals.mean()
    colors = ['#F44336' if v > avg else '#4CAF50' for v in vals]
    axes[i].barh(race_order, vals.values, color=colors)
    axes[i].axvline(x=avg, color='black', linestyle='--',
                    alpha=0.7, label=f'Average ({avg:.1f}%)')
    axes[i].set_title(title)
    axes[i].set_xlabel('Prevalence (%)')
    axes[i].legend(fontsize=8)
    for j, v in enumerate(vals.values):
        axes[i].text(v+0.3, j, f'{v:.1f}%', va='center', fontsize=8)

plt.suptitle('Health Equity Analysis — NCD Prevalence by Race/Ethnicity\n'
             '(Red=Above Average | Green=Below Average)', fontsize=13)
plt.tight_layout()
plt.savefig('chart3_health_equity.png', dpi=150, bbox_inches='tight')
plt.show()

# Insight: Diabetes prevalence is uniform across racial groups (9-12%).
# The real disparities lie in hypertension (Non-Hispanic Black: 41%)
# and obesity (Non-Hispanic Black and Mexican American: 38-40%).
# Non-Hispanic Asian patients show low obesity yet above-average hypertension
# suggesting different cardiovascular risk pathways independent of BMI.

## 7. Conclusions

### Model Performance

| Model | AUC | Diabetes Recall |
|---|---|---|
| Standard Logistic Regression | 0.822 | 11% |
| Balanced Logistic Regression | 0.821 | 82% |

### Key Risk Factor Findings (Odds Ratios)

| Factor | OR | Interpretation |
|---|---|---|
| Age | 4.67 | Strongest predictor — each SD increase raises odds 4.7x |
| BMI | 1.79 | Each SD increase raises diabetes odds 79% |
| Systolic BP | 0.90 | Slight protective effect — likely confounded by treatment |
| Poverty Ratio | 0.87 | Higher income slightly protective |
| Gender (Male) | 0.72 | Males have lower odds in this model |

### Health Equity Findings

| NCD | Highest Burden Group | Lowest Burden Group | Gap |
|---|---|---|---|
| Diabetes | Other Hispanic (12.1%) | Other/Multiracial (9.2%) | 2.9pp |
| Hypertension | Non-Hispanic Black (41.0%) | Mexican American (30.7%) | 10.3pp |
| Obesity | Non-Hispanic Black (40.4%) | Non-Hispanic Asian (13.8%) | 26.6pp |

### African Market Relevance
NCD burden is rising rapidly in Sub-Saharan Africa — Ghana's NCD prevalence mirrors the trajectory of US minority populations 20 years ago. This framework applies directly to community health screening programs and primary care NCD management in resource-limited settings.

---

**Data Source:** CDC NHANES 2017-2018  
**Analysis by:** Afriyie Karikari Bempah, PharmD | [LinkedIn](https://linkedin.com/in/afriyiekarikaribempah) | [GitHub](https://github.com/akbempah1)